In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score

DATA_DIR = Path("/kaggle/input/competitions/ai-generated-text-detection-hw3-ucu-ai-2026")
BASE_DIR = Path("/kaggle/input/notebooks/oleksandrtsybrivskyi/01-eda-split-baseline")
MDEB_DIR = Path("/kaggle/input/notebooks/oleksandrtsybrivskyi/02-transformer-training")

CLASSES = ["human", "fusion", "ai"]
LABEL2ID = {c: i for i, c in enumerate(CLASSES)}

train = pd.read_csv(DATA_DIR / "train.csv")
splits = pd.read_parquet(BASE_DIR / "folds.parquet")
base_oof = pd.read_parquet(BASE_DIR / "baseline_oof_fold0.parquet")
mdeb_oof = pd.read_parquet(MDEB_DIR / "mdeberta_oof_fold0.parquet")

assert (train["id"].to_numpy() == splits["id"].to_numpy()).all()
train["fold"] = splits["fold"].to_numpy()
train["y"] = train["label"].map(LABEL2ID).astype(np.int64)

va = train[train["fold"] == 0].reset_index(drop=True)
assert (va["id"].to_numpy() == base_oof["id"].to_numpy()).all(), "Baseline OOF ID mismatch!"
assert (va["id"].to_numpy() == mdeb_oof["id"].to_numpy()).all(), "mDeBERTa OOF ID mismatch!"
assert (va["y"].to_numpy() == mdeb_oof["y_true"].to_numpy()).all(), "Label mapping mismatch!"

print("Baseline OOF cols:", list(base_oof.columns))
print("mDeBERTa OOF cols:", list(mdeb_oof.columns))

va["pred_base"] = base_oof["y_pred"].to_numpy()
va["pred_mdeb"] = mdeb_oof["y_pred"].to_numpy()
va["n_words"] = va["text"].str.split().str.len()

# 1. Confusion Matrix (mDeBERTa)
cm = pd.DataFrame(
    confusion_matrix(va["y"], va["pred_mdeb"], labels=[0, 1, 2]),
    index=[f"true_{c}" for c in CLASSES],
    columns=[f"pred_{c}" for c in CLASSES],
)
print("\n--- Confusion Matrix (mDeBERTa, Val Fold 0) ---")
print(cm)

# 2. Взаємодоповнюваність із TF-IDF
mdeb_err = va["pred_mdeb"] != va["y"]
base_ok = va["pred_base"] == va["y"]
print(
    f"\nmDeBERTa errors: {mdeb_err.sum()} ({mdeb_err.mean()*100:.2f}%) | "
    f"of which TF-IDF is correct: {(mdeb_err & base_ok).sum()} "
    f"({(mdeb_err & base_ok).sum() / mdeb_err.sum() * 100:.1f}%)"
)

def group_metrics(df, col):
    rows = []
    for k, g in df.groupby(col, observed=False):
        y, p = g["y"].to_numpy(), g["pred_mdeb"].to_numpy()
        macro = f1_score(y, p, average="macro", zero_division=0)
        per = f1_score(y, p, average=None, labels=[0, 1, 2], zero_division=0)
        f_to_h = ((y == 1) & (p == 0)).sum() / max(1, (y == 1).sum()) * 100
        h_to_f = ((y == 0) & (p == 1)).sum() / max(1, (y == 0).sum()) * 100
        rows.append({
            col: k, "n": len(g), "macro_f1": macro,
            "f1_H": per[0], "f1_F": per[1], "f1_A": per[2],
            "F->H %": f_to_h, "H->F %": h_to_f,
        })
    return pd.DataFrame(rows).round(4)

# 3. Розбивка по мові, домену та довжині (~350 слів ≈ 512 токенів)
lang_col = next(c for c in ("language", "lang") if c in va.columns)
print(f"\n--- Metrics by {lang_col} ---")
print(group_metrics(va, lang_col).to_string(index=False))

print("\n--- Metrics by domain ---")
print(group_metrics(va, "domain").sort_values("macro_f1").to_string(index=False))

va["len_bin"] = pd.cut(
    va["n_words"],
    bins=[0, 200, 350, 550, 900, 10**9],
    labels=["<=200w", "201-350w (~512t)", "351-550w", "551-900w", ">900w"],
)
print("\n--- Metrics by text length (words) ---")
print(group_metrics(va, "len_bin").to_string(index=False))

Baseline OOF cols: ['id', 'y_true', 'y_pred', 'score_human', 'score_fusion', 'score_ai']
mDeBERTa OOF cols: ['id', 'y_true', 'y_pred', 'logit_human', 'logit_fusion', 'logit_ai']

--- Confusion Matrix (mDeBERTa, Val Fold 0) ---
             pred_human  pred_fusion  pred_ai
true_human        11920         1417       52
true_fusion        1359        15556       53
true_ai              21           41    24171

mDeBERTa errors: 2943 (5.39%) | of which TF-IDF is correct: 1905 (64.7%)

--- Metrics by lang ---
 lang     n  macro_f1   f1_H   f1_F   f1_A  F->H %  H->F %
de-DE 10330    0.9211 0.8644 0.9034 0.9957   5.116 18.4118
en-EN 44260    0.9387 0.9005 0.9190 0.9967   8.782  8.4701

--- Metrics by domain ---
                domain     n  macro_f1   f1_H   f1_F   f1_A  F->H %  H->F %
             gutenberg  1529    0.7678 0.6038 0.7020 0.9975 17.6471 45.1613
          arxiv_papers  2850    0.7961 0.4746 0.9138 0.9998  2.0468 66.2651
      euro_court_cases  5575    0.8709 0.7652 0.8475 1.000

In [2]:
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

ID2LABEL = {i: c for i, c in enumerate(CLASSES)}

def extract_scores(df):
    for prefix in ("logit_", "score_", "dec_", "prob_", "p_"):
        cols = [f"{prefix}{c}" for c in CLASSES]
        if all(c in df.columns for c in cols):
            return df[cols].to_numpy(dtype=np.float64), cols
    cols = [
        next(c for c in df.columns if cls in c.lower() and c not in ("id", "label", "y_true", "y_pred"))
        for cls in CLASSES
    ]
    return df[cols].to_numpy(dtype=np.float64), cols

def fast_macro_f1(y_true, y_pred):
    cm = np.bincount(y_true * 3 + y_pred, minlength=9).reshape(3, 3)
    tp = np.diag(cm)
    denom = cm.sum(axis=0) + cm.sum(axis=1)
    return float(np.mean(np.where(denom > 0, 2.0 * tp / denom, 0.0)))

def fit_multipliers(probs, y_true):
    # w_ai фіксуємо = 1.0 (масштабно-інваріантно для argmax)
    best_f1, best_w = -1.0, (1.0, 1.0)
    for w0 in np.linspace(0.5, 1.6, 23):
        for w1 in np.linspace(0.5, 1.6, 23):
            preds = (probs * np.array([w0, w1, 1.0])).argmax(axis=1)
            score = fast_macro_f1(y_true, preds)
            if score > best_f1:
                best_f1, best_w = score, (w0, w1)
    # Точний пошук навколо оптимуму
    w0_c, w1_c = best_w
    for w0 in np.linspace(max(0.2, w0_c - 0.1), w0_c + 0.1, 21):
        for w1 in np.linspace(max(0.2, w1_c - 0.1), w1_c + 0.1, 21):
            preds = (probs * np.array([w0, w1, 1.0])).argmax(axis=1)
            score = fast_macro_f1(y_true, preds)
            if score > best_f1:
                best_f1, best_w = score, (w0, w1)
    return np.array([best_w[0], best_w[1], 1.0])

base_te = pd.read_parquet(BASE_DIR / "baseline_test_preds.parquet")
mdeb_te = pd.read_parquet(MDEB_DIR / "mdeberta_test_preds.parquet")
assert (base_te["id"].to_numpy() == mdeb_te["id"].to_numpy()).all(), "Test ID mismatch!"

S_base_va, base_cols = extract_scores(base_oof)
S_base_te, _ = extract_scores(base_te)
L_mdeb_va, mdeb_cols = extract_scores(mdeb_oof)
L_mdeb_te, _ = extract_scores(mdeb_te)
print(f"Detected score cols | Baseline: {base_cols} | mDeBERTa: {mdeb_cols}")

y_va = va["y"].to_numpy()
P_mdeb_va = softmax(L_mdeb_va, axis=1)
X_meta_va = np.hstack([L_mdeb_va, S_base_va])
X_meta_te = np.hstack([L_mdeb_te, S_base_te])

# Внутрішній 5-fold CV на fold 0 (без витоку)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_mdeb_thr = np.zeros(len(y_va), dtype=np.int64)
oof_blend_prob = np.zeros((len(y_va), 3), dtype=np.float64)
oof_blend_thr = np.zeros(len(y_va), dtype=np.int64)

te_blend_prob = np.zeros((len(mdeb_te), 3), dtype=np.float64)
fold_mults_mdeb, fold_mults_blend = [], []

for tr_idx, ev_idx in skf.split(X_meta_va, y_va):
    # A. Лише тюнінг порогів для mDeBERTa
    w_m = fit_multipliers(P_mdeb_va[tr_idx], y_va[tr_idx])
    fold_mults_mdeb.append(w_m)
    oof_mdeb_thr[ev_idx] = (P_mdeb_va[ev_idx] * w_m).argmax(axis=1)

    # B. Мета-модель (LogReg поверх логітів mDeBERTa + скорів TF-IDF)
    meta = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=1000))
    meta.fit(X_meta_va[tr_idx], y_va[tr_idx])
    p_tr = meta.predict_proba(X_meta_va[tr_idx])
    p_ev = meta.predict_proba(X_meta_va[ev_idx])
    oof_blend_prob[ev_idx] = p_ev
    te_blend_prob += meta.predict_proba(X_meta_te) / skf.n_splits

    # C. Бленд + тюнінг порогів
    w_b = fit_multipliers(p_tr, y_va[tr_idx])
    fold_mults_blend.append(w_b)
    oof_blend_thr[ev_idx] = (p_ev * w_b).argmax(axis=1)

results = [
    ("1. TF-IDF baseline", va["pred_base"].to_numpy()),
    ("2. mDeBERTa raw (ref)", va["pred_mdeb"].to_numpy()),
    ("3. mDeBERTa + multipliers (CV)", oof_mdeb_thr),
    ("4. mDeBERTa + TF-IDF blend (CV)", oof_blend_prob.argmax(axis=1)),
    ("5. Blend + multipliers (CV)", oof_blend_thr),
]

rows = []
for name, pred in results:
    per = f1_score(y_va, pred, average=None, labels=[0, 1, 2])
    rows.append({
        "model": name,
        "macro_f1": f1_score(y_va, pred, average="macro"),
        "f1_H": per[0], "f1_F": per[1], "f1_A": per[2],
    })
res_df = pd.DataFrame(rows).round(4)
print("\n--- Validation Fold 0 Comparison (Nested CV, no leakage) ---")
print(res_df.to_string(index=False))

mean_w_b = np.mean(fold_mults_blend, axis=0)
print(f"\nMean blend multipliers [H, F, A]: {np.round(mean_w_b, 4)}")

# Збереження сабмішну, якщо найкращий варіант перевищує референс
best_row = res_df.iloc[res_df["macro_f1"].idxmax()]
ref_f1 = res_df.loc[res_df["model"] == "2. mDeBERTa raw (ref)", "macro_f1"].iloc[0]

if best_row["macro_f1"] > ref_f1:
    sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
    if "Blend + multipliers" in best_row["model"]:
        te_preds = (te_blend_prob * mean_w_b).argmax(axis=1)
    elif "blend" in best_row["model"].lower():
        te_preds = te_blend_prob.argmax(axis=1)
    else:
        te_preds = (softmax(L_mdeb_te, axis=1) * np.mean(fold_mults_mdeb, axis=0)).argmax(axis=1)

    pred_map = pd.Series(te_preds, index=mdeb_te["id"]).map(ID2LABEL)
    sub = pd.DataFrame({"id": sample_sub["id"], "label": sample_sub["id"].map(pred_map)})
    assert len(sub) == 25_000 and (sub["id"].to_numpy() == sample_sub["id"].to_numpy()).all()
    assert sub["label"].notna().all() and set(sub["label"].unique()) == set(CLASSES)
    sub.to_csv("/kaggle/working/submission_blend_mdeb_tfidf.csv", index=False)
    print(f"\nSaved submission_blend_mdeb_tfidf.csv (best: {best_row['model']}, +{best_row['macro_f1'] - ref_f1:.4f} vs ref)")
    print((sub["label"].value_counts(normalize=True) * 100).round(2).reindex(CLASSES))
else:
    print("\nNo improvement over mDeBERTa raw — submission not written.")

Detected score cols | Baseline: ['score_human', 'score_fusion', 'score_ai'] | mDeBERTa: ['logit_human', 'logit_fusion', 'logit_ai']

--- Validation Fold 0 Comparison (Nested CV, no leakage) ---
                          model  macro_f1   f1_H   f1_F   f1_A
             1. TF-IDF baseline    0.8883 0.8336 0.8451 0.9862
          2. mDeBERTa raw (ref)    0.9351 0.8933 0.9155 0.9966
 3. mDeBERTa + multipliers (CV)    0.9361 0.8967 0.9151 0.9966
4. mDeBERTa + TF-IDF blend (CV)    0.9544 0.9261 0.9396 0.9974
    5. Blend + multipliers (CV)    0.9543 0.9265 0.9391 0.9974

Mean blend multipliers [H, F, A]: [1.424 1.168 1.   ]

Saved submission_blend_mdeb_tfidf.csv (best: 4. mDeBERTa + TF-IDF blend (CV), +0.0193 vs ref)
label
human     25.44
fusion    23.19
ai        51.37
Name: proportion, dtype: float64


In [3]:
%%writefile /kaggle/working/train_mdeberta_long.py
import argparse
import math
import os
import time
from contextlib import nullcontext
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import torch.nn as nn
from sklearn.metrics import classification_report, f1_score
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.distributed import DistributedSampler
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_cosine_schedule_with_warmup,
)
import gc

DATA_DIR = Path("/kaggle/input/competitions/ai-generated-text-detection-hw3-ucu-ai-2026")
BASE_DIR = Path("/kaggle/input/notebooks/oleksandrtsybrivskyi/01-eda-split-baseline")
OUT_DIR = Path("/kaggle/working")

MODEL_NAME = "microsoft/mdeberta-v3-base"
CLASSES = ["human", "fusion", "ai"]
LABEL2ID = {c: i for i, c in enumerate(CLASSES)}
ID2LABEL = {i: c for i, c in enumerate(CLASSES)}


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class LongTextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len: int = 1024, labels=None, orig_indices=None):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.labels = labels
        self.orig_indices = orig_indices if orig_indices is not None else np.arange(len(texts))
        self.cls_id = tokenizer.cls_token_id
        self.sep_id = tokenizer.sep_token_id
        assert self.cls_id is not None and self.sep_id is not None

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int):
        ids = self.tokenizer.encode(self.texts[idx], add_special_tokens=False)
        if len(ids) <= self.max_len - 2:
            input_ids = [self.cls_id] + ids + [self.sep_id]
        else:
            # head + [SEP] + mid + [SEP] + tail (без фальшивого шва посеред речення)
            seg_budget = self.max_len - 4
            h = seg_budget // 3
            m = seg_budget // 3
            t = seg_budget - h - m
            mid_start = (len(ids) - m) // 2
            input_ids = (
                [self.cls_id]
                + ids[:h]
                + [self.sep_id]
                + ids[mid_start : mid_start + m]
                + [self.sep_id]
                + ids[-t:]
                + [self.sep_id]
            )
        item = {"idx": int(self.orig_indices[idx]), "input_ids": input_ids}
        if self.labels is not None:
            item["label"] = int(self.labels[idx])
        return item


def make_collate_fn(pad_id: int, max_len: int):
    def collate_fn(batch):
        max_l = max(len(x["input_ids"]) for x in batch)
        max_l = min(max_len, ((max_l + 7) // 8) * 8)

        bs = len(batch)
        input_ids = torch.full((bs, max_l), pad_id, dtype=torch.long)
        attention_mask = torch.zeros((bs, max_l), dtype=torch.long)
        idxs = torch.empty(bs, dtype=torch.long)

        for i, item in enumerate(batch):
            seq = item["input_ids"]
            L = len(seq)
            input_ids[i, :L] = torch.tensor(seq, dtype=torch.long)
            attention_mask[i, :L] = 1
            idxs[i] = item["idx"]

        out = {"idx": idxs, "input_ids": input_ids, "attention_mask": attention_mask}
        if "label" in batch[0]:
            out["labels"] = torch.tensor([x["label"] for x in batch], dtype=torch.long)
        return out

    return collate_fn


@torch.no_grad()
def predict_ddp(model, loader, n_total: int, device, rank: int, world_size: int):
    model.eval()
    local_idxs, local_logits = [], []

    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        local_idxs.append(batch["idx"].numpy())
        local_logits.append(logits.float().cpu().numpy())

    local_idxs = np.concatenate(local_idxs, axis=0)
    local_logits = np.concatenate(local_logits, axis=0)

    gathered = [None] * world_size
    dist.all_gather_object(gathered, (local_idxs, local_logits))

    if rank != 0:
        return None

    full_logits = np.zeros((n_total, len(CLASSES)), dtype=np.float32)
    seen = np.zeros(n_total, dtype=bool)
    for g_idx, g_log in gathered:
        full_logits[g_idx] = g_log
        seen[g_idx] = True

    assert seen.all(), f"Missing predictions for {(~seen).sum()} rows!"
    assert np.isfinite(full_logits).all(), "Non-finite logits detected!"
    return full_logits


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--smoke_test", action="store_true")
    parser.add_argument("--epochs", type=float, default=1.5)
    parser.add_argument("--batch_size", type=int, default=4)
    parser.add_argument("--eval_batch_size", type=int, default=16)
    parser.add_argument("--grad_accum", type=int, default=4)
    parser.add_argument("--max_len", type=int, default=1024)
    parser.add_argument("--lr", type=float, default=2e-5)
    parser.add_argument("--weight_decay", type=float, default=0.01)
    parser.add_argument("--warmup_ratio", type=float, default=0.08)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--evals_per_epoch", type=int, default=2)
    args = parser.parse_args()

    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    device = torch.device(f"cuda:{local_rank}")
    torch.cuda.set_device(device)
    set_seed(args.seed + rank)

    # 1. Дані та спліт
    train_df = pd.read_csv(DATA_DIR / "train.csv")
    test_df = pd.read_csv(DATA_DIR / "test.csv")
    splits_df = pd.read_parquet(BASE_DIR / "folds.parquet")
    assert (train_df["id"].to_numpy() == splits_df["id"].to_numpy()).all()

    train_df["fold"] = splits_df["fold"].to_numpy()
    train_df["y"] = train_df["label"].map(LABEL2ID).astype(np.int64)

    tr_df = train_df[train_df["fold"] != 0].reset_index(drop=True)
    va_df = train_df[train_df["fold"] == 0].reset_index(drop=True)
    full_tr_len, full_va_len, full_te_len = len(tr_df), len(va_df), len(test_df)

    if args.smoke_test:
        tr_df = tr_df.sample(n=1000, random_state=args.seed).reset_index(drop=True)
        va_df = va_df.sample(n=500, random_state=args.seed).reset_index(drop=True)
        test_df = test_df.head(500).reset_index(drop=True)

    if rank == 0:
        mode = "SMOKE TEST" if args.smoke_test else "FULL RUN"
        eff_bs = args.batch_size * args.grad_accum * world_size
        print(
            f"[{mode}] GPUs={world_size} | max_len={args.max_len} (head+mid+tail+[SEP]) | "
            f"per_gpu_bs={args.batch_size} | grad_accum={args.grad_accum} | eff_bs={eff_bs} | "
            f"epochs={args.epochs} | lr={args.lr}"
        )
        print(f"Rows -> Train: {len(tr_df)} | Val: {len(va_df)} | Test: {len(test_df)}")

    # 2. Токенізатор і DataLoader-и (val/test сортуємо за довжиною для швидкого інференсу)
    if rank != 0:
        dist.barrier()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if rank == 0:
        dist.barrier()
    tokenizer.model_max_length = 1_000_000
    collate_fn = make_collate_fn(tokenizer.pad_token_id, args.max_len)

    va_order = np.argsort(va_df["text"].str.len().to_numpy())
    te_order = np.argsort(test_df["text"].str.len().to_numpy())

    tr_ds = LongTextDataset(tr_df["text"].tolist(), tokenizer, args.max_len, tr_df["y"].to_numpy())
    va_ds = LongTextDataset(
        va_df["text"].iloc[va_order].tolist(), tokenizer, args.max_len, None, orig_indices=va_order
    )
    te_ds = LongTextDataset(
        test_df["text"].iloc[te_order].tolist(), tokenizer, args.max_len, None, orig_indices=te_order
    )
    del train_df, splits_df
    tr_df = tr_df[["id"]]
    va_df = va_df[["id", "y"]]
    test_df = test_df[["id"]]
    gc.collect()

    tr_sampler = DistributedSampler(tr_ds, num_replicas=world_size, rank=rank, shuffle=True, seed=args.seed)
    va_sampler = DistributedSampler(va_ds, num_replicas=world_size, rank=rank, shuffle=False)
    te_sampler = DistributedSampler(te_ds, num_replicas=world_size, rank=rank, shuffle=False)

    tr_loader = DataLoader(
        tr_ds, batch_size=args.batch_size, sampler=tr_sampler,
        num_workers=2, pin_memory=True, collate_fn=collate_fn, drop_last=False
    )
    va_loader = DataLoader(
        va_ds, batch_size=args.eval_batch_size, sampler=va_sampler,
        num_workers=2, pin_memory=True, collate_fn=collate_fn, drop_last=False
    )
    te_loader = DataLoader(
        te_ds, batch_size=args.eval_batch_size, sampler=te_sampler,
        num_workers=2, pin_memory=True, collate_fn=collate_fn, drop_last=False
    )

    # 3. Модель (майстер-ваги у FP32 для AMP GradScaler)
    if rank != 0:
        dist.barrier()
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(CLASSES))
    if rank == 0:
        dist.barrier()
    model = model.float()
    assert all(p.dtype == torch.float32 for p in model.parameters()), "Master weights must be FP32!"
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.to(device)
    model = DDP(model, device_ids=[local_rank], output_device=local_rank, find_unused_parameters=False)

    no_decay = ("bias", "LayerNorm.weight", "LayerNorm.bias")
    grouped_params = [
        {
            "params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            "weight_decay": args.weight_decay,
        },
        {
            "params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
        },
    ]
    optimizer = torch.optim.AdamW(grouped_params, lr=args.lr, eps=1e-6)

    steps_per_epoch = math.ceil(len(tr_loader) / args.grad_accum)
    total_opt_steps = max(1, int(math.ceil(steps_per_epoch * args.epochs)))
    warmup_steps = int(total_opt_steps * args.warmup_ratio)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_opt_steps)

    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda", init_scale=4096.0)

    eval_interval = max(1, steps_per_epoch // args.evals_per_epoch)
    best_f1 = -1.0
    best_oof_logits = None
    best_ckpt_path = OUT_DIR / ("mdeberta_long_smoke.pt" if args.smoke_test else "mdeberta_long_best_tmp.pt")

    # 4. Цикл тренування
    global_opt_step = 0
    nan_skips = 0
    total_eval_time = 0.0
    last_eval_time = 0.0
    t_train_start = time.time()
    n_epochs_int = math.ceil(args.epochs)

    for epoch in range(n_epochs_int):
        tr_sampler.set_epoch(epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        log_steps = 0
        t_log = time.time()

        for step, batch in enumerate(tr_loader, start=1):
            is_sync = (step % args.grad_accum == 0) or (step == len(tr_loader))
            ctx = nullcontext() if is_sync else model.no_sync()

            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            with ctx:
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                    loss = criterion(logits.float(), labels) / args.grad_accum
                scaler.scale(loss).backward()

            running_loss += loss.item() * args.grad_accum
            log_steps += 1

            if is_sync:
                scale_before = scaler.get_scale()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

                if scaler.get_scale() < scale_before:
                    nan_skips += 1
                else:
                    scheduler.step()

                global_opt_step += 1

                if rank == 0 and (global_opt_step % max(1, min(200, eval_interval)) == 0 or global_opt_step == 10):
                    dt = time.time() - t_log
                    lr_now = scheduler.get_last_lr()[0]
                    vram = torch.cuda.max_memory_allocated(device) / 1024**3
                    print(
                        f"Ep {epoch + 1} | step {global_opt_step}/{total_opt_steps} | "
                        f"loss={running_loss / log_steps:.4f} | lr={lr_now:.2e} | "
                        f"vram_peak={vram:.2f}GB | skips={nan_skips} | {dt:.1f}s",
                        flush=True,
                    )
                    running_loss, log_steps, t_log = 0.0, 0, time.time()

                do_eval = (global_opt_step % eval_interval == 0) or (global_opt_step >= total_opt_steps)
                if do_eval:
                    t_ev = time.time()
                    val_logits = predict_ddp(model, va_loader, len(va_ds), device, rank, world_size)
                    last_eval_time = time.time() - t_ev
                    total_eval_time += last_eval_time
                    if rank == 0:
                        val_preds = val_logits.argmax(axis=1)
                        va_y = va_df["y"].to_numpy()
                        macro_f1 = f1_score(va_y, val_preds, average="macro")
                        per_cls = f1_score(va_y, val_preds, average=None, labels=[0, 1, 2])
                        print(
                            f">>> [EVAL step {global_opt_step}/{total_opt_steps}] "
                            f"Macro-F1={macro_f1:.4f} (H={per_cls[0]:.4f}, F={per_cls[1]:.4f}, A={per_cls[2]:.4f}) "
                            f"| eval_time={last_eval_time:.1f}s",
                            flush=True,
                        )
                        if macro_f1 > best_f1:
                            best_f1 = macro_f1
                            best_oof_logits = val_logits.copy()
                            torch.save(model.module.state_dict(), best_ckpt_path)
                    dist.barrier()
                    model.train()

                if global_opt_step >= total_opt_steps:
                    break

    pure_train_sec = (time.time() - t_train_start) - total_eval_time

    # 5. Завантаження найкращого чекпоінта та інференс на test
    dist.barrier()
    state = torch.load(best_ckpt_path, map_location=device, weights_only=True)
    model.module.load_state_dict(state)
    t_te = time.time()
    te_logits = predict_ddp(model, te_loader, len(te_ds), device, rank, world_size)
    te_elapsed = time.time() - t_te

    if rank == 0:
        vram_peak = torch.cuda.max_memory_allocated(device) / 1024**3
        print(f"\n=== FINISHED | Best Val Macro-F1={best_f1:.4f} | Peak VRAM={vram_peak:.2f}GB | NaN skips={nan_skips} ===")

        if args.smoke_test:
            est_tr_1ep = (pure_train_sec / args.epochs) * (full_tr_len / len(tr_df)) / 60.0
            est_va_1x = last_eval_time * (full_va_len / len(va_df)) / 60.0
            est_te = te_elapsed * (full_te_len / len(test_df)) / 60.0
            print(
                f"[ETA PROJECTION] 1 full epoch train ≈ {est_tr_1ep:.1f} min | "
                f"1 val eval ≈ {est_va_1x:.1f} min | test inference ≈ {est_te:.1f} min"
            )
            print(
                f"[TOTAL ESTIMATE] 1.5 epochs (3 evals) ≈ {(1.5 * est_tr_1ep + 3 * est_va_1x + est_te) / 60:.2f} h | "
                f"2.0 epochs (4 evals) ≈ {(2.0 * est_tr_1ep + 4 * est_va_1x + est_te) / 60:.2f} h"
            )
            if best_ckpt_path.exists():
                best_ckpt_path.unlink()
        else:
            va_y = va_df["y"].to_numpy()
            va_preds = best_oof_logits.argmax(axis=1)
            print("\n--- Classification Report (Val Fold 0) ---")
            print(classification_report(va_y, va_preds, target_names=CLASSES, digits=4))

            # Зберігаємо ваги в fp16 (~550 MB) для Bonus xAI
            fp16_state = {k: v.detach().to(torch.float16).cpu() for k, v in model.module.state_dict().items()}
            torch.save(fp16_state, OUT_DIR / "mdeberta_long_best_fp16.pt")
            if best_ckpt_path.exists():
                best_ckpt_path.unlink()

            oof_df = pd.DataFrame({
                "id": va_df["id"].to_numpy(),
                "y_true": va_y,
                "y_pred": va_preds,
                "logit_human": best_oof_logits[:, 0],
                "logit_fusion": best_oof_logits[:, 1],
                "logit_ai": best_oof_logits[:, 2],
            })
            te_preds = te_logits.argmax(axis=1)
            te_out = pd.DataFrame({
                "id": test_df["id"].to_numpy(),
                "y_pred": te_preds,
                "logit_human": te_logits[:, 0],
                "logit_fusion": te_logits[:, 1],
                "logit_ai": te_logits[:, 2],
            })
            oof_df.to_parquet(OUT_DIR / "mdeberta_long_oof_fold0.parquet", index=False)
            te_out.to_parquet(OUT_DIR / "mdeberta_long_test_preds.parquet", index=False)

            sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
            pred_map = te_out.set_index("id")["y_pred"].map(ID2LABEL)
            sub = pd.DataFrame({"id": sample_sub["id"], "label": sample_sub["id"].map(pred_map)})
            assert len(sub) == 25_000 and (sub["id"].to_numpy() == sample_sub["id"].to_numpy()).all()
            assert sub["label"].notna().all() and set(sub["label"].unique()) == set(CLASSES)

            sub_path = OUT_DIR / "submission_mdeberta_long.csv"
            sub.to_csv(sub_path, index=False)
            print(f"Saved: mdeberta_long_best_fp16.pt, mdeberta_long_oof_fold0.parquet, mdeberta_long_test_preds.parquet, {sub_path.name}")
            print("\n--- Test Predicted Class % ---")
            print((sub["label"].value_counts(normalize=True) * 100).round(2).reindex(CLASSES))

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing /kaggle/working/train_mdeberta_long.py


In [4]:
!torchrun --nproc_per_node=2 --master_port=29502 /kaggle/working/train_mdeberta_long.py --epochs 1 --max_len 1024

W1121 21:06:54.637000 56 torch/distributed/run.py:851] 
W1121 21:06:54.637000 56 torch/distributed/run.py:851] *****************************************
W1121 21:06:54.637000 56 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1121 21:06:54.637000 56 torch/distributed/run.py:851] *****************************************
[FULL RUN] GPUs=2 | max_len=1024 (head+mid+tail+[SEP]) | per_gpu_bs=4 | grad_accum=4 | eff_bs=32 | epochs=1.0 | lr=2e-05
Rows -> Train: 217762 | Val: 54590 | Test: 25000
config.json: 100%|█████████████████████████████| 579/579 [00:00<00:00, 1.92MB/s]
tokenizer_config.json: 100%|██████████████████| 52.0/52.0 [00:00<00:00, 144kB/s]

spm.model: downloading bytes: ████████████████████████      | 3.45MB            
spm.model: downloading bytes: ██████████████████████████████| 3.99MB,  